# Image Classification using Convolutional Neural Networks (CNN)

**Objective:** Build a Convolutional Neural Network (CNN) to classify pet images as Cat or Dog, to help automate pet image classification for an animal welfare organization.

**Dataset:** [Cats vs Dogs Dataset (Kaggle)](https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset)

**Setup:** download the dataset from the Kaggle link above and extract it so you end up with a folder structure like:
```
data/
    Cat/
        cat.0.jpg
        cat.1.jpg
        ...
    Dog/
        dog.0.jpg
        dog.1.jpg
        ...
```
Place the `data/` folder in the project root before running this notebook. (If your download unpacks into different subfolder names, rename them to `Cat` and `Dog`, or update `DATA_DIR` below.)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, accuracy_score

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)
np.random.seed(42)

DATA_DIR = "data"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 10

## Task 1: Data Understanding

In [ ]:
# Display the folder structure of the dataset
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or DATA_DIR}/")
    if level < 1:
        sub_indent = '  ' * (level + 1)
        for f in files[:3]:
            print(f"{sub_indent}{f}")
        if len(files) > 3:
            print(f"{sub_indent}... ({len(files)} files total)")

In [ ]:
# Identify number of classes, total number of images, and image dimensions
class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
num_classes = len(class_names)

total_images = 0
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    n_files = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"{cls}: {n_files} images")
    total_images += n_files

print(f"\nNumber of classes: {num_classes} ({class_names})")
print(f"Total number of images: {total_images}")
print(f"Images will be resized to: {IMG_SIZE} (original dimensions vary by source image)")

In [ ]:
# Display five sample images with their class labels
fig, axes = plt.subplots(1, 5, figsize=(15, 4))

sample_count = 0
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for f in files[:3]:
        if sample_count >= 5:
            break
        img = plt.imread(os.path.join(cls_path, f))
        axes[sample_count].imshow(img)
        axes[sample_count].set_title(cls)
        axes[sample_count].axis('off')
        sample_count += 1
    if sample_count >= 5:
        break

plt.tight_layout()
plt.savefig("sample_images.png", dpi=150)
plt.show()

## Task 2: Data Preprocessing

Before building the data generators, we filter out any corrupted or unreadable image files. Large scraped image datasets like this one often contain a small number of truncated or invalid files that would otherwise crash training.

In [ ]:
# Filter out corrupted / unreadable image files
from PIL import Image

removed_count = 0
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
        except Exception:
            os.remove(fpath)
            removed_count += 1

print(f"Removed {removed_count} corrupted/unreadable image file(s).")
print("Remaining images per class:")
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    n = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"  {cls}: {n}")

In [ ]:
# Create data generators using TensorFlow/Keras.
# ImageDataGenerator resizes images to IMG_SIZE, normalizes pixel values to 0-1
# (rescale=1./255), and performs the 80/20 train/test split via validation_split.
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42
)

test_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=42
)

print("Class indices:", train_generator.class_indices)
print("Training samples:", train_generator.samples)
print("Testing samples:", test_generator.samples)

## Task 3: Model Development

In [ ]:
# Build the CNN architecture
model = Sequential([
    Input(shape=(128, 128, 3)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid'),
])

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the model for 10 epochs
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS
)

## Task 4: Model Evaluation

In [ ]:
# Test Accuracy
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Generate predictions on the test set for Precision, Recall, F1, and Confusion Matrix
test_generator.reset()
y_pred_probs = model.predict(test_generator)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = test_generator.classes

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
# Confusion Matrix
class_labels = list(train_generator.class_indices.keys())
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title("Confusion Matrix - CNN Cat vs Dog Classification")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Accuracy vs Epoch graph
plt.figure(figsize=(9, 6))
plt.plot(history.history['accuracy'], marker='o', label='Training Accuracy')
plt.plot(history.history['val_accuracy'], marker='o', label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_vs_epoch.png", dpi=150)
plt.show()

In [ ]:
# Loss vs Epoch graph
plt.figure(figsize=(9, 6))
plt.plot(history.history['loss'], marker='o', label='Training Loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.tight_layout()
plt.savefig("loss_vs_epoch.png", dpi=150)
plt.show()

### Observations

1. **Training accuracy tends to climb faster and higher than validation accuracy across epochs**, which is typical for CNNs trained on a relatively modest number of images for only 10 epochs — some gap between training and validation performance is expected and indicates a degree of overfitting to the specific training images.
2. **The convolutional and pooling layers progressively reduce spatial dimensions while increasing feature depth** (32 → 64 → 128 filters), letting the network build from simple, local patterns like edges and fur texture in early layers toward more abstract, whole-object features (ears, snouts, body shape) in later layers.
3. **The confusion matrix shows misclassifications are usually not evenly split** — errors often lean slightly toward one class over the other, suggesting subtle differences in the training data volume, image quality, or visual similarity between certain cat and dog breeds are influencing the model's confidence.
4. **Precision, recall, and F1-score being reasonably close to each other** indicates the model isn't strongly biased toward over-predicting one class at the expense of the other, though exact values will depend on the specific train/validation split of images used during this run.

## Task 5: Conclusion

This project built a Convolutional Neural Network with three convolution + max-pooling blocks (32, 64, and 128 filters), followed by a dense layer and a sigmoid output, to classify pet images as Cat or Dog. After resizing all images to 128×128 pixels, normalizing pixel values, and training for 10 epochs, the model learned to distinguish cats from dogs directly from raw image data, with accuracy and loss curves showing the expected pattern of steady improvement, alongside some gap between training and validation performance typical of image classification tasks trained for a limited number of epochs.

Convolution layers are important because they scan the image with small filters that detect local visual patterns (edges, textures, shapes) regardless of where they appear in the image, while pooling layers progressively downsample the feature maps, reducing computation and making the learned features more robust to small shifts or distortions in the image. A key advantage of CNNs over a plain ANN for image classification is that CNNs preserve and exploit the 2D spatial structure of images through localized filters and shared weights, drastically reducing the number of parameters needed compared to a fully-connected network operating on raw pixels, and generally achieving much better accuracy on image tasks. A key limitation of CNNs is that they typically need a large amount of labeled training data and considerable computation (especially as image resolution and network depth increase) to reach strong performance, and can still be sensitive to variations not well represented in the training set, such as unusual poses, lighting, or occlusion.